In [1]:
%pip install transformers bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA H200 NVL


In [3]:
import os

folder_path = "Gemma/a.LLM_Tables"

folder_path_LLM_statements = "Gemma/b.LLM_Inferences"

folder_path_python_code = "Gemma/c.checking_statements"

folder_path_python_output_checking_statements = "Gemma/d.checking_statements_output"

In [4]:
import torch
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "google/gemma-4-31B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

In [ ]:
prompt0 = [
    {
        "role": "system",
        "content": "You are a careful researcher who generates clean, realistic synthetic statistical tables."
    },
    {
        "role": "user",
        "content": """Generate exactly one synthetic statistical table in valid CSV format.

Requirements:
1. The table must have between 5 and 10 columns.
2. The table must contain exactly 15 data rows, plus 1 header row.
3. Invent realistic column names. Include a mix of:
   - numeric columns (integers and/or floats)
   - Numeric columns must contain only plain numbers (no $, commas, or % symbols)
   - categorical columns
4. Make the data look plausible for a real-world dataset.
5. Ensure values are varied across rows.
6. All rows must have the same number of columns as the header.
7. Do not include commas inside cell values.
8. Do not include explanations, markdown, code fences, titles, or any text before or after the CSV.
9. Output only the CSV table.

The dataset can be about any realistic domain such as bhealth, education, business, transportation, or demographics."""
    },
]

In [ ]:
from transformers import pipeline

prompt = tokenizer.apply_chat_template(
    prompt0,
    tokenize=False,
    add_generation_prompt=True
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

generation = generator(
    prompt,
    do_sample=False,
    max_new_tokens=5000,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

print(generation[0]["generated_text"])